# 🧪 Test 3 — Step-Level PRM on Gemma 2B's OWN Outputs

## Why this test is the real one

| | Test 1 | Test 2 | **Test 3 (this)** |
|---|---|---|---|
| Labels | Solution-level proxy | PRM800K (GPT-4 steps) | **Step-level on Gemma 2B outputs** |
| Problem | Too easy | Wrong distribution | **Neither problem** |
| Result | 89.5% (inflated) | 66.3% (deflated) | **True signal** |

## What this notebook does
1. Loads Gemma 2B and generates solutions on 60 GSM8K problems
2. **You manually label** which step first goes wrong (takes ~15 mins)
3. Trains PRM classifier on Gemma 2B's own embeddings + your step labels
4. Gives the final GO / NO-GO verdict

## Decision thresholds
```
≥ 80%   →  ✅  TRUE GO    — build the full pipeline
70–79%  →  ⚠️  PARTIAL    — use rule-based VCE, skip learned verifier
< 70%   →  ❌  TRUE NO-GO — rethink VCE architecture
```

---
**Before running:** Runtime → Change runtime type → GPU (T4)

## Cell 1 — Auth & Setup

In [1]:
# ── Auth ────────────────────────────────────────────────────────────────────
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKENN'))
print('✅ HuggingFace login successful.')

# ── Mount Drive (saves your work so Colab disconnects don't lose anything) ──
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# SAVE_DIR = '/content/drive/MyDrive/sldvm_test3/'
# os.makedirs(SAVE_DIR, exist_ok=True)
# print(f'✅ Save directory: {SAVE_DIR}')

✅ HuggingFace login successful.


## Cell 2 — Install & Imports

In [2]:
# !pip install -q transformers accelerate datasets scikit-learn

import torch
import re
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID       = 'google/gemma-7b-it'
N_PROBLEMS     = 60      # generates enough wrong solutions to label
MAX_NEW_TOKENS = 400

print(f'Device  : {DEVICE}')
print(f'Model   : {MODEL_ID}')
print(f'Problems: {N_PROBLEMS}')

Device  : cuda
Model   : google/gemma-2b-it
Problems: 60


## Cell 3 — Load Gemma 2B

> ⏱️ Takes ~3–5 minutes. Skips if already loaded.

In [3]:
if 'model' in dir() and model is not None:
    print('✅ Model already in memory — skipping reload.')
else:
    print('📥 Loading Gemma 2B...')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map='auto',
    )
    model.eval()
    print('✅ Model loaded.')

📥 Loading Gemma 2B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ Model loaded.


## Cell 4 — Generate Solutions on GSM8K

> ⏱️ ~15–20 minutes on T4. Cached to Drive after first run.

This generates Gemma 2B's step-by-step solutions.
We need enough **wrong** solutions to label — typically ~50 out of 60 will be wrong at this model size.

In [5]:
# RESULTS_CACHE = SAVE_DIR + 'gsm8k_solutions.json'

COT_PROMPT = """Solve this math problem step by step.
Write EACH step on a new line starting with 'Step N:' where N is the step number.
End your answer with 'Answer: <number>'.

Problem: {problem}

Solution:"""

def extract_answer(text):
    match = re.search(r'Answer:\s*([0-9,\.\-]+)', text)
    if match:
        return match.group(1).replace(',', '').strip()
    nums = re.findall(r'-?\d+\.?\d*', text)
    return nums[-1] if nums else None

def extract_steps(text):
    steps = re.findall(
        r'(Step\s*\d+[:\.].*?)(?=Step\s*\d+[:\.]|Answer:|$)',
        text, re.DOTALL
    )
    return [s.strip() for s in steps if len(s.strip()) > 10]

def get_ground_truth(answer_text):
    nums = re.findall(r'####\s*([0-9,\.\-]+)', answer_text)
    if nums:
        return nums[-1].replace(',', '').strip()
    nums = re.findall(r'-?\d+\.?\d*', answer_text)
    return nums[-1] if nums else None

# ── Load from cache if available ────────────────────────────────────────────
gsm8k = load_dataset('gsm8k', 'main', split=f'test[:{N_PROBLEMS}]')
results = []

for i, example in tqdm(enumerate(gsm8k), total=N_PROBLEMS, desc='Generating'):
    problem     = example['question']
    correct_ans = get_ground_truth(example['answer'])
    prompt      = COT_PROMPT.format(problem=problem)

    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    pred_ans = extract_answer(generated)
    steps    = extract_steps(generated)
    correct  = (pred_ans == correct_ans) if pred_ans and correct_ans else False

    results.append({
        'id':             i,
        'problem':        problem,
        'generated':      generated,
        'steps':          steps,
        'pred_answer':    pred_ans,
        'correct_answer': correct_ans,
        'is_correct':     correct,
        'first_error_step': None,  # you will fill this in Cell 5
    })

    if (i + 1) % 10 == 0:
        acc = sum(r['is_correct'] for r in results) / len(results)
        print(f'  [{i+1}/{N_PROBLEMS}]  running accuracy: {acc:.1%}')

n_correct = sum(r['is_correct'] for r in results)
n_wrong   = len(results) - n_correct
base_acc  = n_correct / len(results)

print(f'\n{"="*50}')
print(f'  Gemma 2B accuracy: {base_acc:.1%}')
print(f'  Correct: {n_correct}   Wrong: {n_wrong}')
print(f'{"="*50}')

Generating:  17%|█▋        | 10/60 [00:48<04:39,  5.59s/it]

  [10/60]  running accuracy: 20.0%


Generating:  33%|███▎      | 20/60 [01:31<02:47,  4.18s/it]

  [20/60]  running accuracy: 20.0%


Generating:  50%|█████     | 30/60 [02:12<01:50,  3.69s/it]

...

  [40/60]  running accuracy: 12.5%


Generating:  83%|████████▎ | 50/60 [03:46<00:40,  4.06s/it]

  [50/60]  running accuracy: 14.0%


Generating: 100%|██████████| 60/60 [04:22<00:00,  4.37s/it]

  [60/60]  running accuracy: 15.0%

  Gemma 2B accuracy: 15.0%
  Correct: 9   Wrong: 51


## Cell 5 — 👀 Print Wrong Solutions for Manual Labelling

**This is the 15-minute manual step. Read carefully.**

For each wrong solution below, you will see the problem, the steps, and the correct answer.

You need to identify: **which step number is where the error FIRST appears?**

### How to label
- Read the steps top to bottom
- Find the FIRST step that does something wrong (wrong calculation, wrong setup, wrong logic)
- Note the step number (1-indexed)

### Examples
```
If Step 1 and 2 are fine but Step 3 multiplies wrong → first_error_step = 3
If Step 1 already sets up the equation wrong          → first_error_step = 1
If you genuinely cannot tell where the error is       → first_error_step = -1 (skip)
```

> **You only need to label ~30 of the wrong solutions.** The rest can be skipped with -1.

In [7]:
# ── Print wrong solutions for you to read ───────────────────────────────────
wrong_results = [r for r in results if not r['is_correct']]
print(f'Total wrong solutions to review: {len(wrong_results)}')
print(f'You only need to label ~30. The rest enter -1.\n')
print('=' * 70)

for r in wrong_results[:35]:   # show first 35 wrong problems
    print(f"\n🔴 Problem ID: {r['id']}")
    print(f"   Problem  : {r['problem']}...")
    print(f"   ✅ Correct answer : {r['correct_answer']}")
    print(f"   ❌ Model answer   : {r['pred_answer']}")
    print(f"   Steps generated  : {len(r['steps'])}")
    for j, step in enumerate(r['steps']):
        print(f"      Step {j+1}: {step[:120]}")
    print('-' * 70)

Total wrong solutions to review: 51
You only need to label ~30. The rest enter -1.


🔴 Problem ID: 2
   Problem  : Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?...
   ✅ Correct answer : 70000
   ❌ Model answer   : -
   Steps generated  : 5
      Step 1: Step 1:**  Start with the original price of the house.
$80,000

**
      Step 2: Step 2:**  Add the amount spent on repairs to the original price.
$80,000 + $50,000 = $130,000

**
      Step 3: Step 3:**  Calculate the increase in value.
150% = 1.5

**
      Step 4: Step 4:**  Multiply the original price by the increase in value.
$80,000 x 1.5 = $120,000

**
      Step 5: Step 5:**  Subtract the original price from the final price to find the profit.
$80,000 - $130,000 = -$50,000

**
----------------------------------------------------------------------

🔴 Problem ID: 3
   Problem  : James decides to ru

## Cell 6 — ✏️ Enter Your Step Labels Here

After reading the output above, fill in the dictionary below.

**Format:** `problem_id: first_error_step_number`

```
Use the problem ID shown above (the number after 'Problem ID:')
Step numbers are 1-indexed (first step = 1)
Use -1 if you cannot determine where the error is
```

> ⚠️ **Aim for at least 25 labeled problems.** More is better but 25 is enough.

In [28]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📝 FILL THIS IN AFTER READING THE PROBLEMS ABOVE
#
# Format:  problem_id : first_error_step_number
#
# Examples:
#   0: 1    ← error at step 1 (wrong setup from the start)
#   1: 3    ← steps 1 & 2 fine, error starts at step 3
#   2: -1   ← cannot tell / skip this one
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# manual_error_labels = {
#     # problem_id : first_error_step
#     # ── paste your labels below this line ──────────────────────────────────

#     # EXAMPLE (delete these and replace with your actual labels):
#     # 0:  2,
#     # 1:  1,
#     # 2:  3,
#     # 3: -1,
#     # ...
# }


# manual_error_labels = {
#     2:  4,
#     3:  2,
#     4:  2,
#     5:  3,
#     6:  4,
#     7:  3,
#     8:  4,
#     9:  2,
#     10: 3,
#     11: 1,
#     12: 2,
#     13: 1,
#     15: 2,
#     16: 1,
#     17: 4,
#     19: 2,
#     20: 2,
#     21: 2,
#     22: -1,   # steps correct, model answer wrong
#     23: 4,
#     24: 2,
#     25: 3,
#     26: 2,
#     27: -1,   # fully correct solution
#     28: 2,
#     29: 1,
#     30: 2,
#     31: 3,
#     33: 2,
#     34: 2,
#     35: 4,
#     36: 2,
#     37: 4,
#     38: 1,
#     39: 3,
# }

manual_error_labels = {
    2:  4,   # Step4: $80k×1.5 — 150% increase means ×2.5, wrong base and multiplier
    3:  1,   # Step1: 3×60=180, ignores "3 times a week" → should be 3×3×60=540
    4:  1,   # Step1: misreads feed setup, gets wrong total from the start
    5:  3,   # Step3: $5+$3=$8 — only sums 2 glasses instead of all 16
    6:  4,   # Step4: 2×4×20=80 (arithmetic wrong) AND doesn't add all 3 cities
    7:  3,   # Step3: 100+40=140 — ignores that download restarts from 0%
    8:  4,   # Step4: 80×4=320 — uses full 4hrs, remaining is only 1.5hrs
    9:  2,   # Step2: "Overtime=45hrs" — overtime is only 45-40=5 hours
   10:  3,   # Step3: uses month1(60) as base, should use month2(180)
   11:  1,   # Step1: 3×$68=$192 — arithmetic error (correct=204)
   12:  2,   # Step2: "7 lemons in 7 years" — misreads annual rate as total
   13:  1,   # Step1: "started with 3" — fabricates number, must work backwards
   15:  2,   # Step2: adds both profits — should pick the better plan ($125)
   16:  1,   # Step1: introduces variable x for 80 miles already stated
   17:  4,   # Step4: truncated but gives $600 instead of $1150×50=$57,500
   19:  2,   # Step2: "2miles/hr=0.5hrs" — problem says it took 1 full hour
   20:  2,   # Step2: calls water content "spilled" — invents concept not in problem
   21:  2,   # Step2: "son was 23" — confuses birth age with current age
   22: -1,   # all steps correct (3+4+0=7), error invisible in output. Skip.
   23:  5,   # Step5: subtracts 4cm (hours) instead of 8cm (from Step3)
   24:  2,   # Step2: "Original=Discount" — nonsensical equation
   25:  4,   # Step4: reuses per-unit prices instead of subtotals from Steps 1-3
   26:  2,   # Step2: "3x=$22.50" — $22.50 is per pair not total, makes x=$7.50
   27: -1,   # all steps correct, 16.00==16 is string mismatch. Skip.
   28:  2,   # Step2: "15 miles between stops" — 15 is stop-to-END, not stop-to-stop
   29:  1,   # Step1: "two high heels=$33" — $33 is per shoe, two=$66
   30:  2,   # Step2: ratio equation ignores total=162 entirely
   31:  3,   # Step3: "x+20=0.5x" — should just compute 0.5×80+20=60
   33:  2,   # Step2: "x=110-30=80" — ignores gold=silver+30 constraint
   34:  2,   # Step2: "x=2-a" — inverted equation (should be Aaron-2=Siobhan)
   35:  4,   # Step4: 4×0.25=10 (wrong) AND "25% more" means ×1.25, not ×0.25
   36:  2,   # Step2: "$5/yogurt" — $5 is per pack of 4, not per yogurt
   37:  2,   # Step2: 13×$15=$180 — arithmetic error (correct=195)
   38:  1,   # Step1: "60miles×3days=180minutes" — wrong operation and wrong units
   39:  3,   # Step3: "skip=x/2" — skip is half of RUN(4x), should be skip=2x
}



# ── Validation ──────────────────────────────────────────────────────────────
valid_labels = {k: v for k, v in manual_error_labels.items() if v != -1}
skipped      = {k: v for k, v in manual_error_labels.items() if v == -1}

print(f'Labels entered  : {len(manual_error_labels)}')
print(f'Valid labels    : {len(valid_labels)}')
print(f'Skipped (-1)    : {len(skipped)}')

if len(valid_labels) < 20:
    print(f'\n⚠️  Only {len(valid_labels)} valid labels.')
    print('   You need at least 20 for a meaningful result.')
    print('   Please label more problems above and re-run this cell.')
else:
    print(f'\n✅ Enough labels to proceed ({len(valid_labels)} valid).')

Labels entered  : 35
Valid labels    : 33
Skipped (-1)    : 2

✅ Enough labels to proceed (33 valid).


## Cell 7 — Build Step-Level Dataset

Using your labels, we now create a **proper step-level dataset**:
- Steps **before** the first error → label `1` (correct)
- Steps **at and after** the first error → label `0` (wrong)

This is the key difference from both previous tests.

In [29]:
# ── Build step-level dataset from your labels ────────────────────────────────
step_texts  = []
step_labels = []
step_meta   = []   # for debugging / spot-check later

# 1. Add steps from CORRECT solutions — all steps labelled 1
correct_results = [r for r in results if r['is_correct']]
for r in correct_results:
    for j, step in enumerate(r['steps']):
        if len(step.strip()) > 10:
            step_texts.append(step)
            step_labels.append(1)
            step_meta.append({'problem_id': r['id'], 'step_idx': j, 'source': 'correct_solution'})

# 2. Add steps from WRONG solutions using your step-level labels
for r in results:
    pid = r['id']
    if pid not in valid_labels:
        continue   # skip unlabelled problems

    first_error = valid_labels[pid]   # 1-indexed step where error starts

    for j, step in enumerate(r['steps']):
        if len(step.strip()) <= 10:
            continue
        step_num = j + 1   # convert to 1-indexed

        if step_num < first_error:
            label = 1   # step is correct (before the error)
        else:
            label = 0   # step is wrong (at or after first error)

        step_texts.append(step)
        step_labels.append(label)
        step_meta.append({
            'problem_id':   pid,
            'step_idx':     j,
            'first_error':  first_error,
            'source':       'wrong_solution',
            'label':        label,
        })

step_labels = np.array(step_labels)
n_pos = step_labels.sum()
n_neg = (step_labels == 0).sum()

print(f'Step-level dataset built:')
print(f'  Total steps     : {len(step_texts)}')
print(f'  Correct (1)     : {n_pos}')
print(f'  Wrong   (0)     : {n_neg}')
print(f'  Class ratio     : {n_pos/len(step_texts):.1%} positive')

if n_pos < 15 or n_neg < 15:
    print('\n⚠️  Very few steps in one class. Consider labelling more problems.')
else:
    print('\n✅ Dataset looks good. Proceeding to embedding.')

Step-level dataset built:
  Total steps     : 175
  Correct (1)     : 76
  Wrong   (0)     : 99
  Class ratio     : 43.4% positive

✅ Dataset looks good. Proceeding to embedding.


## Cell 8 — Balance Classes

Before embedding, we balance the dataset so the classifier isn't biased.

In [30]:
# ── Balance positive and negative steps ─────────────────────────────────────
pos_idx = np.where(step_labels == 1)[0]
neg_idx = np.where(step_labels == 0)[0]
n_each  = min(len(pos_idx), len(neg_idx))

np.random.seed(42)
pos_sample = np.random.choice(pos_idx, n_each, replace=False)
neg_sample = np.random.choice(neg_idx, n_each, replace=False)
balanced_idx = np.concatenate([pos_sample, neg_sample])
np.random.shuffle(balanced_idx)

balanced_texts  = [step_texts[i]  for i in balanced_idx]
balanced_labels = step_labels[balanced_idx]

print(f'Balanced dataset:')
print(f'  Correct steps (1) : {n_each}')
print(f'  Wrong steps   (0) : {n_each}')
print(f'  Total             : {len(balanced_texts)}')

Balanced dataset:
  Correct steps (1) : 76
  Wrong steps   (0) : 76
  Total             : 152


## Cell 9 — Extract Embeddings from Gemma 2B

> ⏱️ ~5–10 minutes depending on dataset size. Cached to Drive.

We use Gemma 2B's last hidden state (mean-pooled) as the embedding.
This is exactly what we did in Test 1 — but now the labels are step-level, not solution-level.

In [31]:
print(f'Embedding {len(balanced_texts)} steps...')

BATCH      = 8
MAX_LENGTH = 128
embeddings = []

for i in tqdm(range(0, len(balanced_texts), BATCH), desc='Embedding'):
    batch = balanced_texts[i:i+BATCH]
    inputs = tokenizer(
        batch,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    hidden = outputs.hidden_states[-1]           # (batch, seq, 2048)
    mask   = inputs['attention_mask'].unsqueeze(-1).float()
    emb    = (hidden * mask).sum(1) / mask.sum(1)  # mean pool
    embeddings.extend(emb.cpu().float().numpy())

X = np.array(embeddings)
y = balanced_labels

print(f'\n  Correct (1): {y.sum()}  |  Wrong (0): {(y==0).sum()}')

Embedding 152 steps...


Embedding: 100%|██████████| 19/19 [00:02<00:00,  6.95it/s]


  Correct (1): 76  |  Wrong (0): 76


## Cell 10 — Train & Evaluate PRM Classifier

In [32]:
from sklearn.metrics import accuracy_score, classification_report

# ── Train / test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Training on {len(X_train)} steps, testing on {len(X_test)} steps...')

clf = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced')
clf.fit(X_train, y_train)

y_pred       = clf.predict(X_test)
prm_accuracy = accuracy_score(y_test, y_pred)

print(f'\n{"="*55}')
print(f'  PRM Step-Level Accuracy : {prm_accuracy:.1%}')
print(f'{"="*55}')
print('\nDetailed Classification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=['Wrong step (0)', 'Correct step (1)']
))

# ── Save classifier for pipeline use later ──────────────────────────────────
# pickle.dump(clf, open(SAVE_DIR + 'prm_classifier_test3.pkl', 'wb'))
print(f'✅ complete')

Training on 114 steps, testing on 38 steps...

  PRM Step-Level Accuracy : 73.7%

Detailed Classification Report:
                  precision    recall  f1-score   support

  Wrong step (0)       0.71      0.79      0.75        19
Correct step (1)       0.76      0.68      0.72        19

        accuracy                           0.74        38
       macro avg       0.74      0.74      0.74        38
    weighted avg       0.74      0.74      0.74        38

✅ complete


## Cell 11 — 🚦 Final Verdict

In [33]:
print('\n' + '='*62)
print('  TEST 3 — STEP-LEVEL PRM — FINAL VERDICT')
print('='*62)
print(f'  Model                    : {MODEL_ID}')
print(f'  Problems generated       : {len(results)}')
print(f'  Problems manually labeled: {len(valid_labels)}')
print(f'  Balanced steps in dataset: {len(X)}')
print(f'  Training steps           : {len(X_train)}')
print(f'  Test steps               : {len(X_test)}')
print()
print(f'  Gemma 2B base accuracy   : {base_acc:.1%}')
print(f'  PRM step-level accuracy  : {prm_accuracy:.1%}')
print()

if prm_accuracy >= 0.80:
    verdict = '✅  TRUE GO'
    colour  = '\033[92m'  # green
    detail  = (
        'Gemma 2B hidden states reliably encode step quality\n'
        '  using YOUR step-level labels on its OWN outputs.\n\n'
        '  ➜ Build the full SLD-VM pipeline.\n'
        '  ➜ VCE with learned PRM is viable.\n'
        '  ➜ Use this classifier (saved to Drive) in Stage 4.'
    )
elif prm_accuracy >= 0.70:
    verdict = '⚠️  PARTIAL GO'
    colour  = '\033[93m'  # yellow
    detail  = (
        'Weak but real signal. Hidden states partially encode\n'
        '  step quality, but not reliably enough for learned VCE.\n\n'
        '  ➜ Build the pipeline WITHOUT a learned VCE.\n'
        '  ➜ Use rule-based arithmetic checks for VCE instead.\n'
        '  ➜ RGC branching + majority vote is still your main gain.'
    )
else:
    verdict = '❌  TRUE NO-GO'
    colour  = '\033[91m'  # red
    detail  = (
        'Hidden states do not encode step correctness meaningfully.\n'
        '  Learned VCE is not viable for Gemma 2B.\n\n'
        '  ➜ Still build LTG + RGC (no VCE needed).\n'
        '  ➜ Use only rule-based arithmetic/logic checks for VCE.\n'
        '  ➜ Or switch base model to Phi-3 Mini / Qwen2.5-1.5B.'
    )

RESET = '\033[0m'
print(f'  VERDICT: {colour}{verdict}{RESET}')
print()
print(f'  {detail}')
print('='*62)

# ── Save full summary to Drive ───────────────────────────────────────────────
summary = {
    'test':               'Test 3 — Step-Level PRM on Gemma 2B Own Outputs',
    'model':              MODEL_ID,
    'n_problems':         len(results),
    'n_labeled':          len(valid_labels),
    'n_balanced_steps':   len(X),
    'base_accuracy':      base_acc,
    'prm_accuracy':       prm_accuracy,
    'verdict':            verdict,
    'threshold_go':       0.80,
    'threshold_partial':  0.70,
}



  TEST 3 — STEP-LEVEL PRM — FINAL VERDICT
  Model                    : google/gemma-2b-it
  Problems generated       : 60
  Problems manually labeled: 33
  Balanced steps in dataset: 152
  Training steps           : 114
  Test steps               : 38

  Gemma 2B base accuracy   : 15.0%
  PRM step-level accuracy  : 73.7%

  VERDICT: ⚠️  PARTIAL GO

  Weak but real signal. Hidden states partially encode
  step quality, but not reliably enough for learned VCE.

  ➜ Build the pipeline WITHOUT a learned VCE.
  ➜ Use rule-based arithmetic checks for VCE instead.
  ➜ RGC branching + majority vote is still your main gain.


## Cell 12 — Spot-Check: Does the PRM flag real errors?

Visual sanity check — does the PRM actually flag the steps YOU labeled as wrong?

If it does, you can trust the result. If it flags random steps, the signal is noise.

In [15]:
def embed_single(text, max_length=128):
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=max_length
    ).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    hidden = outputs.hidden_states[-1]
    mask   = inputs['attention_mask'].unsqueeze(-1).float()
    emb    = (hidden * mask).sum(1) / mask.sum(1)
    return emb.squeeze(0).cpu().float().numpy()

# ── Pick up to 5 labeled wrong problems and see how PRM scores each step ─────
labeled_wrong = [
    r for r in results
    if not r['is_correct'] and r['id'] in valid_labels
][:5]

print('--- Spot-check: PRM scores on labeled wrong problems ---\n')
all_correct_flags = []

for r in labeled_wrong:
    pid         = r['id']
    first_error = valid_labels[pid]

    print(f"Problem {pid}: {r['problem'][:80]}...")
    print(f"  ✅ Correct answer : {r['correct_answer']}")
    print(f"  ❌ Model answer   : {r['pred_answer']}")
    print(f"  📌 First error at step: {first_error} (your label)")

    step_correct = []
    for j, step in enumerate(r['steps']):
        if len(step.strip()) < 10:
            continue
        step_num     = j + 1
        true_label   = 1 if step_num < first_error else 0
        emb          = embed_single(step).reshape(1, -1)
        prob_wrong   = clf.predict_proba(emb)[0][0]
        pred_label   = 0 if prob_wrong > 0.5 else 1
        flag         = '⚠ FLAGGED' if prob_wrong > 0.5 else '  ok     '
        match        = '✓' if pred_label == true_label else '✗'
        step_correct.append(pred_label == true_label)

        print(f"    Step {step_num} [{flag} | wrong-prob={prob_wrong:.2f}] {match}  {step[:65]}")

    all_correct_flags.extend(step_correct)
    step_acc = sum(step_correct) / len(step_correct) if step_correct else 0
    print(f"  Step accuracy on this problem: {step_acc:.0%}\n")

overall_spot = sum(all_correct_flags) / len(all_correct_flags) if all_correct_flags else 0
print(f'\n{"="*55}')
print(f'  Spot-check accuracy: {overall_spot:.1%}')
print(f'  (Should be close to the test accuracy above: {prm_accuracy:.1%})')
print(f'{"="*55}')

--- Spot-check: PRM scores on labeled wrong problems ---

Problem 2: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts...
  ✅ Correct answer : 70000
  ❌ Model answer   : -
  📌 First error at step: 4 (your label)
    Step 1 [  ok      | wrong-prob=0.01] ✓  Step 1:**  Start with the original price of the house.
$80,000

*
    Step 2 [  ok      | wrong-prob=0.04] ✓  Step 2:**  Add the amount spent on repairs to the original price.
    Step 3 [  ok      | wrong-prob=0.03] ✓  Step 3:**  Calculate the increase in value.
150% = 1.5

**
    Step 4 [  ok      | wrong-prob=0.43] ✗  Step 4:**  Multiply the original price by the increase in value.

    Step 5 [⚠ FLAGGED | wrong-prob=0.99] ✓  Step 5:**  Subtract the original price from the final price to fi
  Step accuracy on this problem: 80%

Problem 3: James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  ...
  ✅ Correct answer : 540
  ❌ Model answer   : 9360
  📌 First error at step: 2 (yo